# HumanLM Local Inspection

Notebook ini aman untuk laptop lokal karena **tidak load model penuh**.

Fokus:
- baca `config.json`
- inspect tokenizer
- lihat chat template
- pahami arsitektur dasar dari metadata model

In [ ]:
import json
from transformers import AutoConfig, AutoTokenizer

MODEL_NAME = "snap-stanford/humanlm-opinion"
print(MODEL_NAME)

## 1. Config

In [ ]:
cfg = AutoConfig.from_pretrained(MODEL_NAME)

summary = {
    "model_type": getattr(cfg, "model_type", None),
    "architectures": getattr(cfg, "architectures", None),
    "hidden_size": getattr(cfg, "hidden_size", None),
    "intermediate_size": getattr(cfg, "intermediate_size", None),
    "num_hidden_layers": getattr(cfg, "num_hidden_layers", None),
    "num_attention_heads": getattr(cfg, "num_attention_heads", None),
    "num_key_value_heads": getattr(cfg, "num_key_value_heads", None),
    "vocab_size": getattr(cfg, "vocab_size", None),
    "max_position_embeddings": getattr(cfg, "max_position_embeddings", None),
    "rope_theta": getattr(cfg, "rope_theta", None),
    "torch_dtype": str(getattr(cfg, "torch_dtype", None)),
}

summary

In [ ]:
print(json.dumps(cfg.to_dict(), indent=2, default=str)[:5000])

## 2. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_info = {
    "tokenizer_class": tokenizer.__class__.__name__,
    "vocab_size": tokenizer.vocab_size,
    "pad_token": tokenizer.pad_token,
    "eos_token": tokenizer.eos_token,
    "bos_token": tokenizer.bos_token,
    "chat_template_exists": bool(getattr(tokenizer, "chat_template", None)),
    "special_tokens_map": tokenizer.special_tokens_map,
}

tokenizer_info

In [ ]:
if getattr(tokenizer, "chat_template", None):
    print(tokenizer.chat_template[:3000])
else:
    print("No chat template found.")

## 3. Tokenisasi Prompt Kecil

Sel ini aman, karena belum load model.

In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

print({k: tuple(v.shape) for k, v in inputs.items()})
print(inputs["input_ids"][0][:50])

## 4. Catatan

Kalau ingin melihat forward pass, hidden states, atau generate output model, gunakan notebook khusus VM.